# Equity Research Notebook

## Libraries

In [47]:
import wrds
import numpy as np
import pandas as pd
import datetime as dt
import yfinance as yf
from scipy.stats import linregress
from scipy.optimize import minimize

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_colwidth', None)

## Tickers

In [26]:
tickers = ['GEV', 'WDC', 'WELL', 'GLW', 'HWM', 'PM', 'NEM', 'DUOL', 'NFLX']

## Analysis Configuration

In [27]:
start_year = 2020
weight_cap = 0.45
interval = '1d'

macro_factors = ['^VIX', 'GC=F', '^TNX']
treasury_yield = '^FVX'
market_index = '^GSPC'

if interval == '1d':
    assume_trading_days = 252
elif interval == '1wk':
    assume_trading_days = 52
elif interval == '1mo':
    assume_trading_days = 12

console_formating = 180

## Analysis Toggle

In [ ]:
MARKET_TABLE = True
PORTFOLIO_TABLE = True
FUNDAMENTALS_TABLE = True

MACRO_TABLE = True
COMPS_TABLE = True
HIST_MULTIPLES_TABLE = True
COMPS_SECTOR_MEDIAN_TABLE = True

TICKERS_FILTER = True

## Table Format Function

In [29]:
def _print_df_with_bars(df):
    try:
        from tabulate import tabulate
        print(tabulate(df, headers='keys', tablefmt='github', showindex=True, stralign='center', numalign='center', disable_numparse=True))
        return
    except Exception:
        pass
    try:
        print(df.to_markdown(tablefmt='github'))
        return
    except Exception:
        pass
    import re
    s = df.to_string()
    s = re.sub(r' {2,}', ' | ', s)
    print(s)

## Fetch Market Data and Calculate Returns 

In [30]:
today = dt.datetime.today()
end = today - dt.timedelta(days=1)
start = dt.datetime(start_year, 1, 1)
price_tickers = [market_index] + tickers

df_prices = yf.download(price_tickers, start=start, end=end, interval=interval, auto_adjust=False).dropna()
rf_last = yf.download(treasury_yield, start=start, end=end, interval=interval, auto_adjust=False)['Adj Close'].iloc[-1]
rf_annual = (float(rf_last.iloc[0]) if isinstance(rf_last, pd.Series) else float(rf_last)) / 100
rf = rf_annual / assume_trading_days

df_log_returns = np.log(df_prices['Adj Close']).diff().dropna()
df_mean_returns = df_log_returns.mean()
df_variance_returns = df_log_returns.var()
df_stdev_returns = df_log_returns.std()

index_returns = df_log_returns[market_index]
beta_dict = {}
alpha_dict = {}

for stock in tickers:
    stock_returns = df_log_returns[stock]
    excess_market = index_returns - rf
    excess_stock = stock_returns - rf
    slope, intercept, r_value, p_value, std_err = linregress(excess_market, excess_stock)
    beta_dict[stock] = slope
    alpha_dict[stock] = intercept

annual_mean_returns = np.exp(df_mean_returns * assume_trading_days) - 1
annual_variance_returns = df_variance_returns * assume_trading_days
annual_stdev_returns = df_stdev_returns * np.sqrt(assume_trading_days)
annual_alpha_dict = {stock: alpha * assume_trading_days for stock, alpha in alpha_dict.items()}
annual_capm_dict = {stock: rf + beta_dict[stock] * (annual_mean_returns[market_index] - rf) for stock in beta_dict.keys()}

stock_tickers = tickers.copy()
df_stock_returns = df_log_returns[stock_tickers]
excess_returns = df_stock_returns - df_stock_returns.mean()
n_obs = len(excess_returns)
variance_covariance_matrix = (excess_returns.T @ excess_returns) / (n_obs - 1)
annualized_variance_covariance_matrix = variance_covariance_matrix * assume_trading_days
correlation_matrix = df_stock_returns.corr()

try:
    df_macro_prices = yf.download(macro_factors, start=start, end=end, interval=interval, auto_adjust=False).dropna()
except Exception:
    df_macro_prices = pd.DataFrame()

if not df_macro_prices.empty:
    df_macro_returns = np.log(df_macro_prices['Adj Close']).diff().dropna()
else:
    df_macro_returns = pd.DataFrame()

if not df_macro_returns.empty:
    combined_returns = pd.concat([df_stock_returns, df_macro_returns], axis=1)
    correlation_macro = combined_returns.corr().loc[stock_tickers, macro_factors]
    correlation_macro.columns = ['VIX', 'Gold', 'Interest']
else:
    correlation_macro = pd.DataFrame(index=stock_tickers, columns=['VIX', 'Gold', 'Interest'])

[*********************100%***********************]  10 of 10 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  3 of 3 completed


## Modern Portfolio Theory

In [31]:
def portfolio_variance(weights, cov_matrix):
    return weights @ cov_matrix @ weights

def negative_sharpe_ratio(weights, returns, cov_matrix, risk_free_rate):
    port_return = np.sum(weights * returns)
    port_variance = weights @ cov_matrix @ weights
    port_sd = np.sqrt(port_variance)
    sharpe = (port_return - risk_free_rate) / port_sd if port_sd != 0 else 0
    return -sharpe

def least_correlation_objective(weights, corr_matrix):
    return weights @ corr_matrix @ weights

def negative_return(weights, returns):
    return -np.sum(weights * returns)

annualized_returns = np.array([annual_mean_returns[ticker] for ticker in stock_tickers])
initial_weights = np.array([1 / len(stock_tickers)] * len(stock_tickers))
constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
bounds = tuple((0, weight_cap) for _ in stock_tickers)

result = minimize(
    portfolio_variance,
    initial_weights,
    args=(annualized_variance_covariance_matrix,),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)

min_var_weights = result.x
min_var_portfolio_variance = result.fun
min_var_portfolio_sd = np.sqrt(min_var_portfolio_variance)
min_var_portfolio_return = np.sum(min_var_weights * annualized_returns)
min_var_sharpe_ratio = (min_var_portfolio_return - rf) / min_var_portfolio_sd if min_var_portfolio_sd != 0 else 0

min_var_df = pd.DataFrame({
    'Stock': stock_tickers,
    'Weight': min_var_weights,
    'var(P)': min_var_portfolio_variance,
    'SD(P)': min_var_portfolio_sd,
    'E[R_P]': min_var_portfolio_return,
    'S_P': min_var_sharpe_ratio
}).set_index('Stock')

result_tangent = minimize(
    negative_sharpe_ratio,
    initial_weights,
    args=(annualized_returns, annualized_variance_covariance_matrix, rf),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)

tangent_weights = result_tangent.x
tangent_portfolio_variance = tangent_weights @ annualized_variance_covariance_matrix @ tangent_weights
tangent_portfolio_sd = np.sqrt(tangent_portfolio_variance)
tangent_portfolio_return = np.sum(tangent_weights * annualized_returns)
tangent_sharpe_ratio = (tangent_portfolio_return - rf) / tangent_portfolio_sd if tangent_portfolio_sd != 0 else 0

tangent_port_df = pd.DataFrame({
    'Stock': stock_tickers,
    'Weight': tangent_weights,
    'var(P)': tangent_portfolio_variance,
    'SD(P)': tangent_portfolio_sd,
    'E[R_P]': tangent_portfolio_return,
    'S_P': tangent_sharpe_ratio
}).set_index('Stock')

result_least_corr = minimize(
    least_correlation_objective,
    initial_weights,
    args=(correlation_matrix.values,),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)

least_corr_weights = result_least_corr.x
least_corr_portfolio_corr = least_correlation_objective(least_corr_weights, correlation_matrix.values)
least_corr_portfolio_return = np.sum(least_corr_weights * annualized_returns)
least_corr_portfolio_variance = least_corr_weights @ annualized_variance_covariance_matrix @ least_corr_weights
least_corr_portfolio_sd = np.sqrt(least_corr_portfolio_variance)
least_corr_sharpe_ratio = (least_corr_portfolio_return - rf) / least_corr_portfolio_sd if least_corr_portfolio_sd != 0 else 0

least_corr_df = pd.DataFrame({
    'Stock': stock_tickers,
    'Weight': least_corr_weights,
    'Corr(P)': least_corr_portfolio_corr,
    'E[R_P]': least_corr_portfolio_return,
    'var(P)': least_corr_portfolio_variance,
    'SD(P)': least_corr_portfolio_sd,
    'S_P': least_corr_sharpe_ratio
}).set_index('Stock')

result_max_return = minimize(
    negative_return,
    initial_weights,
    args=(annualized_returns,),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)

max_return_weights = result_max_return.x
max_return_portfolio_return = np.sum(max_return_weights * annualized_returns)
max_return_portfolio_variance = max_return_weights @ annualized_variance_covariance_matrix @ max_return_weights
max_return_portfolio_sd = np.sqrt(max_return_portfolio_variance)
max_return_sharpe_ratio = (max_return_portfolio_return - rf) / max_return_portfolio_sd if max_return_portfolio_sd != 0 else 0

max_return_df = pd.DataFrame({
    'Stock': stock_tickers,
    'Weight': max_return_weights,
    'E[R_P]': max_return_portfolio_return,
    'var(P)': max_return_portfolio_variance,
    'SD(P)': max_return_portfolio_sd,
    'S_P': max_return_sharpe_ratio
}).set_index('Stock')

## Summarize Market Metrics and Calculate Momentum

In [32]:
def _get_price_on_or_after(series, dt_obj):
    s = series[series.index >= pd.Timestamp(dt_obj)]
    return s.iloc[0] if not s.empty else series.iloc[0]

total_returns = {stock: (df_prices['Adj Close'][stock].iloc[-1] / df_prices['Adj Close'][stock].iloc[0]) - 1 for stock in stock_tickers}
market_total_return = (df_prices['Adj Close'][market_index].iloc[-1] / df_prices['Adj Close'][market_index].iloc[0]) - 1

start_of_year = dt.datetime(today.year, 1, 1)
ytd_returns = {}
for ticker in stock_tickers:
    series = df_prices['Adj Close'][ticker]
    start_price = _get_price_on_or_after(series, start_of_year)
    ytd_returns[ticker] = (series.iloc[-1] / start_price) - 1

market_start_price = _get_price_on_or_after(df_prices['Adj Close'][market_index], start_of_year)
market_ytd_return = (df_prices['Adj Close'][market_index].iloc[-1] / market_start_price) - 1

latest_prices = {}
market_caps = {}
for ticker in stock_tickers:
    latest_prices[ticker] = df_prices['Adj Close'][ticker].iloc[-1]
    try:
        info = yf.Ticker(ticker).info
        market_caps[ticker] = info.get('marketCap', np.nan)
    except Exception:
        market_caps[ticker] = np.nan

total_market_cap = sum(v for v in market_caps.values() if not pd.isna(v)) if len(market_caps) > 0 else np.nan

summary_data = [{
    'Ticker': 'S&P500',
    'Total Return (%)': market_total_return * 100,
    'YTD Return (%)': market_ytd_return * 100,
    'Mean Return (%)': annual_mean_returns[market_index] * 100,
    'σ (%)': annual_stdev_returns[market_index] * 100,
    'Sharpe Ratio': (annual_mean_returns[market_index] - rf) / annual_stdev_returns[market_index] if annual_stdev_returns[market_index] != 0 else 0,
    'β': 1.0,
    'CAPM (%)': annual_mean_returns[market_index] * 100,
    'α (%)': 0,
    'Market Price': df_prices['Adj Close'][market_index].iloc[-1],
    'MC ($B)': (total_market_cap / 1e9) if not pd.isna(total_market_cap) else np.nan,
    'MC Proportion (%)': 100.0
}]

for stock in stock_tickers:
    mcap = market_caps.get(stock, np.nan)
    pct = (mcap / total_market_cap * 100) if (not pd.isna(mcap) and total_market_cap and total_market_cap > 0) else np.nan
    summary_data.append({
        'Ticker': stock,
        'Total Return (%)': total_returns[stock] * 100,
        'YTD Return (%)': ytd_returns.get(stock, np.nan) * 100,
        'Mean Return (%)': annual_mean_returns[stock] * 100,
        'σ (%)': annual_stdev_returns[stock] * 100,
        'Sharpe Ratio': (annual_mean_returns[stock] - rf) / annual_stdev_returns[stock] if annual_stdev_returns[stock] != 0 else 0,
        'β': beta_dict[stock],
        'CAPM (%)': annual_capm_dict[stock] * 100,
        'α (%)': annual_alpha_dict[stock] * 100,
        'Market Price': latest_prices.get(stock, np.nan),
        'MC ($B)': (mcap / 1e9) if not pd.isna(mcap) else np.nan,
        'MC Proportion (%)': pct
    })

summary_df = pd.DataFrame(summary_data).set_index('Ticker')

momentum_6m = {}
momentum_12m = {}
for ticker in stock_tickers:
    series = df_prices['Adj Close'][ticker]
    try:
        price_now = series.iloc[-1]
        price_6m = series.iloc[-126]
        price_12m = series.iloc[-252]
        momentum_6m[ticker] = (price_now / price_6m) - 1
        momentum_12m[ticker] = (price_now / price_12m) - 1
    except Exception:
        momentum_6m[ticker] = np.nan
        momentum_12m[ticker] = np.nan

summary_df['Momentum_6M'] = pd.Series(momentum_6m) * 100
summary_df['Momentum_12M'] = pd.Series(momentum_12m) * 100

summary_df['PT'] = np.nan
summary_df['ΔPT (%)'] = np.nan

## Fetch WRDS Data

In [33]:
conn = wrds.Connection(wrds_username='tobiasnhh')  # create pgpass file for automated login

def fetch_wrds_price_targets(connection, selected_tickers):
    if not selected_tickers:
        return pd.DataFrame(columns=['tic', 'price_target', 'target_date', 'source'])

    normalized_tickers = sorted({str(t).upper().strip() for t in selected_tickers if str(t).strip()})
    ticker_sql = ', '.join(["'{}'".format(t.replace("'", "''")) for t in normalized_tickers])

    query_attempts = [
        f"""
            SELECT UPPER(TRIM(oftic)) AS tic,
                   statpers AS target_date,
                   meanptg AS price_target
            FROM ibes.ptgsum
            WHERE UPPER(TRIM(oftic)) IN ({ticker_sql})
              AND meanptg IS NOT NULL
              AND usfirm = 1
              AND measure = 'PTG'
            ORDER BY oftic, statpers DESC
        """,
        f"""
            SELECT UPPER(TRIM(oftic)) AS tic,
                   actdats AS target_date,
                   value AS price_target
            FROM ibes.ptgdet
            WHERE UPPER(TRIM(oftic)) IN ({ticker_sql})
              AND value IS NOT NULL
              AND usfirm = 1
              AND measure = 'PTG'
              AND horizon = 12
            ORDER BY oftic, actdats DESC
        """
    ]

    for idx, query in enumerate(query_attempts, start=1):
        try:
            pt_data = connection.raw_sql(query)
            if pt_data is None or pt_data.empty:
                print(f'[WRDS PT] Query {idx} returned no rows.')
                continue

            pt_data = pt_data.copy()
            pt_data['tic'] = pt_data['tic'].astype(str).str.upper().str.strip()
            pt_data['target_date'] = pd.to_datetime(pt_data['target_date'], errors='coerce')
            pt_data['price_target'] = pd.to_numeric(pt_data['price_target'], errors='coerce')
            pt_data['source'] = 'ibes.ptgsum' if idx == 1 else 'ibes.ptgdet'
            pt_data = pt_data.dropna(subset=['tic', 'price_target']).sort_values(['tic', 'target_date'], ascending=[True, False])
            pt_data = pt_data.drop_duplicates(subset=['tic'], keep='first')

            missing_tickers = sorted(set(normalized_tickers) - set(pt_data['tic'].unique()))
            if missing_tickers:
                print(f"[WRDS PT] Missing targets for: {', '.join(missing_tickers)}")

            return pt_data[['tic', 'price_target', 'target_date', 'source']]
        except Exception as exc:
            print(f'[WRDS PT] Query {idx} failed: {exc}')
            continue

    return pd.DataFrame(columns=['tic', 'price_target', 'target_date', 'source'])

gvkeys = conn.raw_sql(f"""
    SELECT DISTINCT gvkey, tic
    FROM comp.funda
    WHERE tic IN {tuple(tickers)}
""")

gvkey_list = tuple(gvkeys.gvkey.unique())

data = conn.raw_sql(f"""
    SELECT gvkey, tic, datadate, fyear,
        sale, ebit, ni, csho,
        oancf,
        dltt, dlc, ceq
    FROM comp.funda
    WHERE gvkey IN {gvkey_list}
      AND indfmt = 'INDL'
      AND datafmt = 'STD'
      AND popsrc = 'D'
      AND consol = 'C'
      AND fyear >= {start_year - 1}
    ORDER BY gvkey, fyear
""")

data['datadate'] = pd.to_datetime(data['datadate'], errors='coerce')
data = data[data['datadate'].notna()]
data = data[data['datadate'] <= pd.Timestamp.today()]
data = data.sort_values(['tic', 'datadate'])

data['Revenue_Growth'] = data.groupby('tic')['sale'].pct_change(fill_method=None)
data['EBIT_Margin'] = data['ebit'] / data['sale']

_csho = pd.to_numeric(data['csho'], errors='coerce')
data['EPS'] = np.where(_csho.fillna(0) > 0, data['ni'] / _csho, np.nan)
data['EPS_Growth'] = data.groupby('tic')['EPS'].pct_change(fill_method=None)

data['OCF_Growth'] = data.groupby('tic')['oancf'].pct_change(fill_method=None)
data['OCF_Margin'] = data['oancf'] / data['sale']

data['Debt_to_Equity'] = (data['dltt'] + data['dlc']) / data['ceq']

def default_rating(x):
    if pd.isna(x):
        return 'Risky'
    elif x > 0.247:
        return 'Aaa'
    elif x > 0.188:
        return 'Aa'
    elif x > 0.142:
        return 'A'
    elif x > 0.128:
        return 'Baa'
    elif x > 0.118:
        return 'Ba'
    elif x > 0.095:
        return 'B'
    elif x > 0.045:
        return 'C'
    else:
        return 'Risky'

data['Default_Risk'] = data['EBIT_Margin'].apply(default_rating)
data = data[data['fyear'] >= start_year]

metrics = ['Revenue_Growth', 'EBIT_Margin', 'EPS_Growth', 'OCF_Growth', 'OCF_Margin', 'Debt_to_Equity', 'Default_Risk']
final = data[['tic', 'fyear'] + metrics].copy().sort_values(['tic', 'fyear']).set_index(['tic', 'fyear'])

Loading library list...
Done


## Calculate Historical Multiples and Comparables

In [34]:
def _get_latest_non_null(series):
    s = series.dropna()
    return s.iloc[-1] if not s.empty else np.nan

price_long = df_prices['Adj Close'][stock_tickers].copy().stack().reset_index()
price_long.columns = ['date', 'tic', 'price']
price_long['date'] = pd.to_datetime(price_long['date'], errors='coerce')
price_long['tic'] = price_long['tic'].astype(str)
price_long = price_long.dropna(subset=['date', 'tic', 'price']).sort_values(['tic', 'date'])

fund_cols_for_merge = ['tic', 'datadate', 'sale', 'ebit', 'ni', 'csho', 'dltt', 'dlc']
fund_pt = data[fund_cols_for_merge].copy().sort_values(['tic', 'datadate'])
fund_pt['tic'] = fund_pt['tic'].astype(str)

merged_frames = []
for ticker in stock_tickers:
    price_t = price_long[price_long['tic'] == ticker].sort_values('date')
    fund_t = fund_pt[fund_pt['tic'] == ticker].sort_values('datadate').drop(columns=['tic'])
    if price_t.empty or fund_t.empty:
        continue
    merged_t = pd.merge_asof(price_t, fund_t, left_on='date', right_on='datadate', direction='backward')
    merged_t['tic'] = ticker
    merged_frames.append(merged_t)

historical_multiples = pd.concat(merged_frames, ignore_index=True) if merged_frames else pd.DataFrame()
historical_multiples = historical_multiples.dropna(subset=['sale', 'ebit', 'ni', 'csho'])

historical_multiples['EPS'] = np.where(historical_multiples['csho'] > 0, historical_multiples['ni'] / historical_multiples['csho'], np.nan)
historical_multiples['P/E'] = np.where(historical_multiples['EPS'] > 0, historical_multiples['price'] / historical_multiples['EPS'], np.nan)

historical_multiples['shares'] = historical_multiples['csho'] * 1e6
historical_multiples['market_cap'] = historical_multiples['price'] * historical_multiples['shares']
historical_multiples['total_debt'] = (historical_multiples['dltt'] + historical_multiples['dlc']) * 1e6
historical_multiples['enterprise_value'] = historical_multiples['market_cap'] + historical_multiples['total_debt']

historical_multiples['EV/Sales'] = np.where(historical_multiples['sale'] > 0, historical_multiples['enterprise_value'] / (historical_multiples['sale'] * 1e6), np.nan)
historical_multiples['EV/EBIT'] = np.where(historical_multiples['ebit'] > 0, historical_multiples['enterprise_value'] / (historical_multiples['ebit'] * 1e6), np.nan)

sector_map = {}
for ticker in stock_tickers:
    try:
        sector_map[ticker] = yf.Ticker(ticker).info.get('sector', 'N/A')
    except Exception:
        sector_map[ticker] = 'N/A'

historical_multiples['Sector'] = historical_multiples['tic'].map(sector_map).fillna('N/A')

hist_start = pd.Timestamp(dt.datetime(today.year - 1, 1, 1))
historical_multiples = historical_multiples[historical_multiples['date'] >= hist_start]

hist_summary_rows = []
for ticker in stock_tickers:
    sub = historical_multiples[historical_multiples['tic'] == ticker].sort_values('date')
    if sub.empty:
        continue
    current_row = sub.iloc[-1]
    hist_summary_rows.append({
        'Ticker': ticker,
        'P/E_Current': current_row['P/E'],
        'P/E_Median': sub['P/E'].median(skipna=True),
        'P/E_Min': sub['P/E'].min(skipna=True),
        'P/E_Max': sub['P/E'].max(skipna=True),
        'EV/Sales_Current': current_row['EV/Sales'],
        'EV/Sales_Median': sub['EV/Sales'].median(skipna=True),
        'EV/Sales_Min': sub['EV/Sales'].min(skipna=True),
        'EV/Sales_Max': sub['EV/Sales'].max(skipna=True),
        'EV/EBIT_Current': current_row['EV/EBIT'],
        'EV/EBIT_Median': sub['EV/EBIT'].median(skipna=True),
        'EV/EBIT_Min': sub['EV/EBIT'].min(skipna=True),
        'EV/EBIT_Max': sub['EV/EBIT'].max(skipna=True),
        'Sector': current_row['Sector'],
        'EPS_Current': current_row['EPS'],
        'Revenue_Current': current_row['sale'] * 1e6,
        'EBIT_Current': current_row['ebit'] * 1e6,
        'Debt_Current': current_row['total_debt'],
        'Shares_Current': current_row['shares']
    })

historical_multiples_summary = pd.DataFrame(hist_summary_rows)

if not historical_multiples_summary.empty:
    industry_avg = historical_multiples_summary.groupby('Sector')[['P/E_Current', 'EV/Sales_Current', 'EV/EBIT_Current']].mean().rename(columns={
        'P/E_Current': 'P/E_Industry',
        'EV/Sales_Current': 'EV/Sales_Industry',
        'EV/EBIT_Current': 'EV/EBIT_Industry'
    })

    historical_multiples_summary = historical_multiples_summary.merge(industry_avg, left_on='Sector', right_index=True, how='left')
    historical_multiples_summary['P/E_Price'] = historical_multiples_summary['P/E_Industry'] * historical_multiples_summary['EPS_Current']

    implied_ev_sales = historical_multiples_summary['EV/Sales_Industry'] * historical_multiples_summary['Revenue_Current']
    implied_equity_sales = implied_ev_sales - historical_multiples_summary['Debt_Current']
    historical_multiples_summary['EV/Sales_Price'] = np.where(
        historical_multiples_summary['Shares_Current'] > 0,
        implied_equity_sales / historical_multiples_summary['Shares_Current'],
        np.nan
    )

    implied_ev_ebit = historical_multiples_summary['EV/EBIT_Industry'] * historical_multiples_summary['EBIT_Current']
    implied_equity_ebit = implied_ev_ebit - historical_multiples_summary['Debt_Current']
    historical_multiples_summary['EV/EBIT_Price'] = np.where(
        historical_multiples_summary['Shares_Current'] > 0,
        implied_equity_ebit / historical_multiples_summary['Shares_Current'],
        np.nan
    )

historical_multiples_output_cols = ['Ticker', 'P/E_Current', 'P/E_Min', 'P/E_Max', 'EV/Sales_Current', 'EV/Sales_Min', 'EV/Sales_Max', 'EV/EBIT_Current', 'EV/EBIT_Min', 'EV/EBIT_Max', 'P/E_Price', 'EV/Sales_Price', 'EV/EBIT_Price']

if not historical_multiples_summary.empty:
    historical_multiples_table = historical_multiples_summary[historical_multiples_output_cols].set_index('Ticker')
else:
    historical_multiples_table = pd.DataFrame(columns=historical_multiples_output_cols[1:])

wrds_price_targets = fetch_wrds_price_targets(conn, stock_tickers)
wrds_target_map = wrds_price_targets.set_index('tic')['price_target'].to_dict() if not wrds_price_targets.empty else {}

summary_df.loc[:, 'PT'] = np.nan
summary_df.loc[:, 'ΔPT (%)'] = np.nan

for stock in stock_tickers:
    pt_value = wrds_target_map.get(stock)
    latest_price = latest_prices.get(stock, np.nan)
    summary_df.loc[stock, 'PT'] = pt_value if pt_value is not None else np.nan
    if pt_value is not None and not pd.isna(latest_price) and latest_price > 0:
        summary_df.loc[stock, 'ΔPT (%)'] = ((pt_value / latest_price) - 1) * 100

summary_col_order = ['Market Price', 'MC Proportion (%)', 'YTD Return (%)', 'Mean Return (%)', 'Total Return (%)', 'Momentum_6M', 'Momentum_12M', 'σ (%)', 'β', 'Sharpe Ratio', 'α (%)', 'PT', 'ΔPT (%)']
summary_df = summary_df[summary_col_order]

pd.options.display.float_format = '{:.2%}'.format

## Calculate COMPS

In [35]:
latest_fundamentals = data.loc[data.groupby('tic')['datadate'].idxmax()].copy()

comps_data = []
for stock in stock_tickers:
    firm = latest_fundamentals[latest_fundamentals['tic'] == stock]
    if firm.empty:
        continue
    firm = firm.iloc[0]

    try:
        sector = yf.Ticker(stock).info.get('sector', 'N/A')
    except Exception:
        sector = 'N/A'

    revenue = firm['sale'] * 1e6
    ebit = firm['ebit'] * 1e6
    net_income = firm['ni'] * 1e6
    total_debt = (firm['dltt'] + firm['dlc']) * 1e6

    price = latest_prices.get(stock, np.nan)
    market_cap = market_caps.get(stock, np.nan)

    if pd.isna(price) or pd.isna(market_cap):
        continue

    enterprise_value = market_cap + total_debt
    shares = market_cap / price if price > 0 else np.nan

    pe = (price / (net_income / shares)) if shares and net_income > 0 else np.nan
    ev_sales = enterprise_value / revenue if revenue > 0 else np.nan
    ev_ebit = enterprise_value / ebit if ebit > 0 else np.nan

    comps_data.append({
        'Company': stock,
        'Sector': sector,
        'P/E': pe,
        'EV/Sales': ev_sales,
        'EV/EBIT': ev_ebit
    })

comps_numeric = pd.DataFrame(comps_data).set_index('Company')

industry_median = comps_numeric.groupby('Sector')[['P/E', 'EV/Sales', 'EV/EBIT']].median().round(2)

def add_signal(row, column):
    value = row[column]
    median = industry_median.loc[row['Sector'], column]
    if pd.isna(value) or pd.isna(median):
        return ''
    elif value > median:
        return ' (^) '
    elif value < median:
        return ' (⌄) '
    else:
        return ''

comps_display = comps_numeric.copy()
for col in ['P/E', 'EV/Sales', 'EV/EBIT']:
    comps_display[col] = comps_numeric.apply(
        lambda row: f"{round(row[col],2)}{add_signal(row,col)}" if not pd.isna(row[col]) else np.nan,
        axis=1
    )

def add_price_signal(ticker, implied_price):
    market_price = latest_prices.get(ticker, np.nan)
    if pd.isna(implied_price) or pd.isna(market_price):
        return ''
    elif implied_price > market_price:
        return ' (^) '
    elif implied_price < market_price:
        return ' (⌄) '
    else:
        return ''

if not historical_multiples_summary.empty:
    comps_hist = historical_multiples_summary.set_index('Ticker')[['P/E_Median', 'EV/Sales_Median', 'EV/EBIT_Median', 'P/E_Price', 'EV/Sales_Price', 'EV/EBIT_Price']]
    comps_display = comps_display.join(comps_hist, how='left')

    for col in ['P/E_Median', 'EV/Sales_Median', 'EV/EBIT_Median']:
        comps_display[col] = comps_display[col].map(lambda x: f'{x:.2f}' if pd.notna(x) else np.nan)

    for col in ['P/E_Price', 'EV/Sales_Price', 'EV/EBIT_Price']:
        comps_display[col] = comps_display.apply(
            lambda row: f"{row[col]:.2f}{add_price_signal(row.name, row[col])}" if pd.notna(row[col]) else np.nan,
            axis=1
        )
else:
    for col in ['P/E_Median', 'EV/Sales_Median', 'EV/EBIT_Median', 'P/E_Price', 'EV/Sales_Price', 'EV/EBIT_Price']:
        comps_display[col] = np.nan

comps_display = comps_display[['Sector', 'P/E', 'P/E_Median', 'EV/Sales', 'EV/Sales_Median', 'EV/EBIT', 'EV/EBIT_Median', 'P/E_Price', 'EV/Sales_Price', 'EV/EBIT_Price']]

## Portfolio Theory Output

In [36]:
portfolio_data = {
    'Min Variance': [],
    'Tangent': [],
    'Min Correlation': [],
    'Max Return': []
}

portfolio_data['Min Variance'].append(f'{min_var_portfolio_return*100:.2f} %')
portfolio_data['Tangent'].append(f'{tangent_portfolio_return*100:.2f} %')
portfolio_data['Min Correlation'].append(f'{least_corr_portfolio_return*100:.2f} %')
portfolio_data['Max Return'].append(f'{max_return_portfolio_return*100:.2f} %')

portfolio_data['Min Variance'].append(f'{min_var_portfolio_sd*100:.2f} %')
portfolio_data['Tangent'].append(f'{tangent_portfolio_sd*100:.2f} %')
portfolio_data['Min Correlation'].append(f'{least_corr_portfolio_sd*100:.2f} %')
portfolio_data['Max Return'].append(f'{max_return_portfolio_sd*100:.2f} %')

portfolio_data['Min Variance'].append(f'{min_var_portfolio_variance:.2f}')
portfolio_data['Tangent'].append(f'{tangent_portfolio_variance:.2f}')
portfolio_data['Min Correlation'].append(f'{least_corr_portfolio_variance:.2f}')
portfolio_data['Max Return'].append(f'{max_return_portfolio_variance:.2f}')

portfolio_data['Min Variance'].append(f'{min_var_sharpe_ratio:.2f}')
portfolio_data['Tangent'].append(f'{tangent_sharpe_ratio:.2f}')
portfolio_data['Min Correlation'].append(f'{least_corr_sharpe_ratio:.2f}')
portfolio_data['Max Return'].append(f'{max_return_sharpe_ratio:.2f}')

portfolio_data['Min Variance'].append('---')
portfolio_data['Tangent'].append('---')
portfolio_data['Min Correlation'].append('---')
portfolio_data['Max Return'].append('---')

non_zero_mask = []
for i in range(len(stock_tickers)):
    non_zero_mask.append(
        min_var_weights[i] > 1e-6 or tangent_weights[i] > 1e-6 or least_corr_weights[i] > 1e-6 or max_return_weights[i] > 1e-6
    )

for i, stock in enumerate(stock_tickers):
    if not non_zero_mask[i]:
        continue
    portfolio_data['Min Variance'].append(f'{min_var_weights[i]*100:.2f} %')
    portfolio_data['Tangent'].append(f'{tangent_weights[i]*100:.2f} %')
    portfolio_data['Min Correlation'].append(f'{least_corr_weights[i]*100:.2f} %')
    portfolio_data['Max Return'].append(f'{max_return_weights[i]*100:.2f} %')

index_labels = ['E[R]', 'σ', 'Variance', 'Sharpe Ratio', ''] + [f' {stock}' for i, stock in enumerate(stock_tickers) if non_zero_mask[i]]
combined_portfolio_df = pd.DataFrame(portfolio_data, index=index_labels)

<h3 style="color: #ff8c00; font-size: 2em; font-weight: bold; border-bottom: 2px solid #ffffff; padding-bottom: 5px;">
MARKET DATA
</h3>

In [37]:
if MARKET_TABLE:
    #print('\n' + '=' * console_formating)
    print('(Annualized)'.center(console_formating))
    #print('=' * console_formating + '\n')

    summary_display = summary_df.sort_values('MC Proportion (%)', ascending=False).copy()
    for col in summary_display.columns:
        if col in ['Market Price', 'PT']:
            summary_display[col] = summary_display[col].map(lambda x: f'{round(x):.0f}' if pd.notna(x) else np.nan)
        else:
            summary_display[col] = summary_display[col].map(lambda x: f'{x:.1f}' if pd.notna(x) else np.nan)
    _print_df_with_bars(summary_display)

                                                                                    (Annualized)                                                                                    
|  Ticker  |  Market Price  |  MC Proportion (%)  |  YTD Return (%)  |  Mean Return (%)  |  Total Return (%)  |  Momentum_6M  |  Momentum_12M  |  σ (%)  |  β  |  Sharpe Ratio  |  α (%)  |  PT  |  ΔPT (%)  |
|----------|----------------|---------------------|------------------|-------------------|--------------------|---------------|----------------|---------|-----|----------------|---------|------|-----------|
|  S&P500  |      6583      |        100.0        |       -4.0       |       12.0        |        25.4        |      nan      |      nan       |  16.2   | 1.0 |      0.7       |   0.0   | nan  |    nan    |
|   NFLX   |       99       |        27.9         |       8.4        |       26.8        |        60.8        |     -15.1     |      5.5       |  32.9   | 0.9 |      0.8       |  13.2   | 113  |   1

<h3 style="color: #ff8c00; font-size: 2em; font-weight: bold; border-bottom: 2px solid #ffffff; padding-bottom: 5px;">
PORTFOLIO OPTIMIZATION
</h3>

In [ ]:
if PORTFOLIO_TABLE:
    _print_df_with_bars(combined_portfolio_df)

|              |  Min Variance  |  Tangent  |  Min Correlation  |  Max Return  |
|--------------|----------------|-----------|-------------------|--------------|
|     E[R]     |    48.84 %     |  83.40 %  |      54.00 %      |   147.44 %   |
|      σ       |    15.77 %     |  19.87 %  |      21.55 %      |   44.48 %    |
|   Variance   |      0.02      |   0.04    |       0.05        |     0.20     |
| Sharpe Ratio |      3.10      |   4.20    |       2.50        |     3.31     |
|              |      ---       |    ---    |        ---        |     ---      |
|     GEV      |     0.00 %     |  14.32 %  |      4.43 %       |   45.00 %    |
|     WDC      |     2.31 %     |  6.17 %   |      12.12 %      |   45.00 %    |
|     WELL     |    40.70 %     |  35.01 %  |      11.05 %      |    0.00 %    |
|     GLW      |     4.98 %     |  14.48 %  |      7.35 %       |   10.00 %    |
|     HWM      |     3.00 %     |  4.47 %   |      0.24 %       |    0.00 %    |
|      PM      |    27.93 % 

<h3 style="color: #ff8c00; font-size: 2em; font-weight: bold; border-bottom: 2px solid #ffffff; padding-bottom: 5px;">
COMPS TABLE
</h3>

In [ ]:
if COMPS_TABLE:
    if COMPS_SECTOR_MEDIAN_TABLE:
        _print_df_with_bars(industry_median)
        print('\n')
    _print_df_with_bars(comps_display.sort_values(['Sector', 'Company']))
    #print('* Compared to the sector median multiples. \n** Compared to the sector implied price.')

|         Sector         |  P/E   |  EV/Sales  |  EV/EBIT  |
|------------------------|--------|------------|-----------|
|    Basic Materials     | 17.51  |    5.72    |   11.87   |
| Communication Services | 38.11  |    9.64    |   31.23   |
|   Consumer Defensive   | 21.71  |    7.28    |   19.19   |
|      Industrials       | 56.01  |    9.09    |   88.85   |
|      Real Estate       | 150.69 |   15.02    |  416.29   |
|       Technology       | 53.39  |    8.73    |   49.63   |


|  Company  |         Sector         |    P/E     |  P/E_Median  |  EV/Sales  |  EV/Sales_Median  |  EV/EBIT   |  EV/EBIT_Median  |  P/E_Price  |  EV/Sales_Price  |  EV/EBIT_Price  |
|-----------|------------------------|------------|--------------|------------|-------------------|------------|------------------|-------------|------------------|-----------------|
|    NEM    |    Basic Materials     | 17.51 (^)  |    18.67     |  5.72 (⌄)  |       4.61        | 11.87 (⌄)  |      12.44       |   114.05    

***P/E, EV/Sales, EV/EBIT - Company multiple compared to sector median**

***P/E_Price, EV/Sales_Price, EV/EBIT_Price - Compared to comps implied price to company market price**

***Median_multiple compares each tickers to its own median (LY)**

<h3 style="color: #ff8c00; font-size: 2em; font-weight: bold; border-bottom: 2px solid #ffffff; padding-bottom: 5px;">
COMPS ANALYSIS
</h3>

In [ ]:
if HIST_MULTIPLES_TABLE:
    _print_df_with_bars(historical_multiples_table.round(2))

|  Ticker  |  P/E_Current  |  P/E_Min  |  P/E_Max  |  EV/Sales_Current  |  EV/Sales_Min  |  EV/Sales_Max  |  EV/EBIT_Current  |  EV/EBIT_Min  |  EV/EBIT_Max  |  P/E_Price  |  EV/Sales_Price  |  EV/EBIT_Price  |
|----------|---------------|-----------|-----------|--------------------|----------------|----------------|-------------------|---------------|---------------|-------------|------------------|-----------------|
|   GEV    |     49.59     |   34.33   |  128.35   |        6.39        |      2.16      |      6.57      |      131.42       |     91.19     |    309.51     |   1010.74   |     1274.29      |     602.76      |
|   WDC    |     53.87     |   11.63   |   57.88   |        11.2        |      1.4       |      12.0      |       50.05       |     12.59     |    140.54     |   254.84    |      205.61      |     270.79      |
|   WELL   |    150.42     |   81.09   |  159.63   |        15.0        |     11.76      |     18.59      |      415.65       |     78.94     |    437.74   

<h3 style="color: #ff8c00; font-size: 2em; font-weight: bold; border-bottom: 2px solid #ffffff; padding-bottom: 5px;">
FUNDAMENTAL ANALYSIS
</h3>

In [ ]:
if FUNDAMENTALS_TABLE:
    percent_cols = ['Revenue_Growth', 'EBIT_Margin', 'EPS_Growth', 'OCF_Growth', 'OCF_Margin']
    for ticker in final.index.get_level_values(0).unique():
        print(f'{ticker}')
        print('--------------------------')
        fundamentals_display = final.loc[ticker].copy().sort_index(ascending=False)
        for col in percent_cols:
            fundamentals_display[col] = fundamentals_display[col].map(lambda x: f'{x:.2%}' if pd.notna(x) else np.nan)
        fundamentals_display['Debt_to_Equity'] = fundamentals_display['Debt_to_Equity'].map(lambda x: f'{x:.2f}' if pd.notna(x) else np.nan)
        print(fundamentals_display)
        print('\n')

DUOL
--------------------------
      Revenue_Growth EBIT_Margin EPS_Growth OCF_Growth OCF_Margin Debt_to_Equity Default_Risk
fyear                                                                                        
2025          38.71%      13.61%    350.52%     35.83%     37.38%           0.07          Baa
2024          40.84%       8.53%    421.71%     85.86%     38.17%           0.07            C
2023          43.74%      -2.43%   -125.60%    186.29%     28.92%           0.04        Risky
2022          47.34%     -17.56%     -6.06%    485.13%     14.52%           0.05        Risky
2021          55.09%     -22.25%     27.43%    -48.22%      3.66%           0.06        Risky
2020         128.51%      -9.90%     12.86%    722.86%     10.95%          -0.11        Risky


GEV
--------------------------
      Revenue_Growth EBIT_Margin EPS_Growth OCF_Growth OCF_Margin Debt_to_Equity Default_Risk
fyear                                                                                    

<h3 style="color: #ff8c00; font-size: 2em; font-weight: bold; border-bottom: 2px solid #ffffff; padding-bottom: 5px;">
MACRO FACTOR CORRELATION
</h3>

In [ ]:
if MACRO_TABLE:
    if not correlation_macro.empty:
        macro_ytd = {}
        for macro in macro_factors:
            series = df_macro_prices['Adj Close'][macro]
            start_price = _get_price_on_or_after(series, start_of_year)
            ytd_change = (series.iloc[-1] / start_price) - 1
            macro_ytd[macro] = ytd_change * 100

        macro_ytd_formatted = {
            'VIX': f"{macro_ytd['^VIX']:+.0f}%",
            'Gold': f"{macro_ytd['GC=F']:+.0f}%",
            'Interest': f"{macro_ytd['^TNX']:+.0f}%"
        }

        change_row = pd.DataFrame([macro_ytd_formatted], index=['CHANGE'])
        separator_row = pd.DataFrame([['---', '---', '---']], columns=['VIX', 'Gold', 'Interest'], index=[''])
        macro_table = pd.concat([change_row, separator_row, correlation_macro.round(2)])
        _print_df_with_bars(macro_table)
    else:
        print('No macro data available to compute correlations.')

|        |  VIX  |  Gold  |  Interest  |
|--------|-------|--------|------------|
| CHANGE | +65%  |  +8%   |    +3%     |
|        |  ---  |  ---   |    ---     |
|  GEV   | -0.5  |  0.06  |    0.13    |
|  WDC   | -0.44 |  0.22  |    0.09    |
|  WELL  | -0.23 |  0.11  |   -0.22    |
|  GLW   | -0.46 |  0.1   |    0.02    |
|  HWM   | -0.49 |  0.07  |    0.08    |
|   PM   | -0.08 |  0.08  |   -0.17    |
|  NEM   | -0.22 |  0.67  |   -0.06    |
|  DUOL  | -0.36 |  0.0   |    0.1     |
|  NFLX  | -0.37 |  0.07  |    0.05    |


<h3 style="color: #ff8c00; font-size: 2em; font-weight: bold; border-bottom: 2px solid #ffffff; padding-bottom: 5px;">
OPTIMAL TICKERS FILTER
</h3>

In [ ]:
TOP_PERCENTILE = 0.50

def _percentile_score_high(series):
    s = pd.to_numeric(series, errors='coerce')
    if s.notna().sum() == 0:
        return pd.Series(0.0, index=series.index)
    return s.rank(method='average', pct=True).fillna(0.0)

def _percentile_score_low(series):
    s = pd.to_numeric(series, errors='coerce')
    if s.notna().sum() == 0:
        return pd.Series(0.0, index=series.index)
    return (-s).rank(method='average', pct=True).fillna(0.0)

def _top_count(n_items, top_percentile=TOP_PERCENTILE):
    return max(1, int(np.ceil(n_items * top_percentile)))



if TICKERS_FILTER:
    fundamentals_latest = latest_fundamentals.set_index('tic')

    market_df = summary_df.loc[stock_tickers].copy()
    market_df['Score_Return'] = _percentile_score_high(market_df['Mean Return (%)'])
    market_df['Score_Sharpe'] = _percentile_score_high(market_df['Sharpe Ratio'])
    market_df['Score_Alpha'] = _percentile_score_high(market_df['α (%)'])
    market_df['Score_Momentum6M'] = _percentile_score_high(market_df['Momentum_6M'])
    market_df['Score_Momentum12M'] = _percentile_score_high(market_df['Momentum_12M'])
    market_df['Score_Upside'] = _percentile_score_high(market_df['ΔPT (%)'])

    market_df['Market_Score'] = (
        0.22 * market_df['Score_Return']
        + 0.18 * market_df['Score_Sharpe']
        + 0.18 * market_df['Score_Alpha']
        + 0.16 * market_df['Score_Momentum6M']
        + 0.14 * market_df['Score_Momentum12M']
        + 0.12 * market_df['Score_Upside']
    )

    cutoff_market = _top_count(len(market_df), TOP_PERCENTILE)
    top_market = market_df.sort_values('Market_Score', ascending=False).head(cutoff_market)
    top_market_tickers = top_market.index.tolist()

    print(f'Top Market Analysis ({int(TOP_PERCENTILE * 100)}%):')
    print(top_market_tickers)
    print('\n')

    comps_filter_df = comps_numeric.loc[comps_numeric.index.intersection(top_market_tickers)].copy()
    comps_scores = []

    for ticker in comps_filter_df.index:
        row = comps_filter_df.loc[ticker]
        sector = row['Sector']

        if sector not in industry_median.index:
            continue

        sector_med = industry_median.loc[sector]
        pe = row['P/E']
        eps_growth = fundamentals_latest.loc[ticker]['EPS_Growth']
        ev_sales = row['EV/Sales']
        sector_ev_sales = sector_med['EV/Sales']

        peg = np.nan
        if pd.notna(pe) and pd.notna(eps_growth) and eps_growth != 0:
            peg = pe / (eps_growth * 100)

        ev_sales_relative = np.nan
        if pd.notna(ev_sales) and pd.notna(sector_ev_sales) and sector_ev_sales != 0:
            ev_sales_relative = ev_sales / sector_ev_sales

        comps_scores.append({
            'Ticker': ticker,
            'PEG': peg,
            'EV_Sales_Relative': ev_sales_relative
        })

    if len(comps_scores) > 0:
        comps_scores_df = pd.DataFrame(comps_scores).set_index('Ticker')
        comps_scores_df['Score_PEG'] = _percentile_score_low(comps_scores_df['PEG'])
        comps_scores_df['Score_EVSalesRel'] = _percentile_score_low(comps_scores_df['EV_Sales_Relative'])
        comps_scores_df['Score'] = 0.55 * comps_scores_df['Score_PEG'] + 0.45 * comps_scores_df['Score_EVSalesRel']
        cutoff_comps = _top_count(len(comps_scores_df), TOP_PERCENTILE)
        top_comps = comps_scores_df.sort_values('Score', ascending=False).head(cutoff_comps)
        top_comps_tickers = top_comps.index.tolist()
    else:
        top_comps_tickers = []

    print(f'Top COMPS ({int(TOP_PERCENTILE * 100)}% of Market Filter):')
    print(top_comps_tickers)
    print('\n')

    fundamentals_filter = fundamentals_latest.loc[fundamentals_latest.index.intersection(top_comps_tickers)].copy()
    if not fundamentals_filter.empty:
        fundamentals_filter = fundamentals_filter.join(market_df[['Market_Score']], how='left')
        fundamentals_filter['Score_RevenueGrowth'] = _percentile_score_high(fundamentals_filter['Revenue_Growth'])
        fundamentals_filter['Score_EPSGrowth'] = _percentile_score_high(fundamentals_filter['EPS_Growth'])
        fundamentals_filter['Score_OCFGrowth'] = _percentile_score_high(fundamentals_filter['OCF_Growth'])
        fundamentals_filter['Score_EBITMargin'] = _percentile_score_high(fundamentals_filter['EBIT_Margin'])
        fundamentals_filter['Score_DebtToEquity'] = _percentile_score_low(fundamentals_filter['Debt_to_Equity'])
        fundamentals_filter['Score_Market'] = _percentile_score_high(fundamentals_filter['Market_Score'])

        fundamentals_filter['Fundamental_Score'] = (
            0.20 * fundamentals_filter['Score_RevenueGrowth']
            + 0.22 * fundamentals_filter['Score_EPSGrowth']
            + 0.18 * fundamentals_filter['Score_OCFGrowth']
            + 0.20 * fundamentals_filter['Score_EBITMargin']
            + 0.10 * fundamentals_filter['Score_DebtToEquity']
            + 0.10 * fundamentals_filter['Score_Market']
        )

        cutoff_fundamentals = _top_count(len(fundamentals_filter), TOP_PERCENTILE)
        top_fundamentals = fundamentals_filter.sort_values('Fundamental_Score', ascending=False).head(cutoff_fundamentals)
        top_fundamental_tickers = top_fundamentals.index.tolist()
    else:
        top_fundamental_tickers = []

    print(f'Top Fundamentals ({int(TOP_PERCENTILE * 100)}% from Market+COMPS):')
    print(top_fundamental_tickers)
    print('\n')

Top Market Analysis (50%):
['GEV', 'WDC', 'GLW', 'HWM', 'NEM']


Top COMPS (50% of Market Filter):
['NEM', 'GEV', 'WDC']


Top Fundamentals (50% from Market+COMPS):
['NEM', 'GEV']


